In [ ]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"
from ssf.utils.TorchUtils import print_torch_memory, clear_torch_memory
from datasets import load_dataset
import pandas as pd
from convokit import Corpus, download
from pathlib import Path
from ssf.Taxonomy import Taxonomy
from ssf.Constants import *
from ssf.prompt_builders.InferenceGenerationPromptBuilder import InferenceGenerationPromptBuilder
from ssf.prompt_builders.InferenceClassificationPromptBuilder import InferenceClassificationPromptBuilder
from ssf.generation_strategies.configs import ModelConfig, GenerationConfig
from ssf.generation_strategies.VllmGenerationStrategy import VllmGenerationStrategy
from ssf.utils import InferenceUtils
from ssf.Constants import * 

SSF_GENERATOR_HF = "joelmire/llama3.1-8b-it-ssf-generator"  # LoRA adapter on HuggingFace
SSF_CLASSIFIER_HF = "joelmire/llama3.1-8b-it-ssf-classifier"  # LoRA adapter on HuggingFace
BASE_MODEL_HF = "meta-llama/Llama-3.1-8B-Instruct"  # Base model needed for LoRA
NUM_DEMO_STORIES = 10
OUTPUTS_DIR = "outputs"
INF_GEN_PROMPTS_DIR = f"{OUTPUTS_DIR}/inference_generation/prompts"
INF_GEN_OUTPUTS_DIR = f"{OUTPUTS_DIR}/inference_generation/outputs"
INF_CLASS_PROMPTS_DIR = f"{OUTPUTS_DIR}/inference_classification/prompts"
INF_CLASS_OUTPUTS_DIR = f"{OUTPUTS_DIR}/inference_classification/outputs"

Path(INF_GEN_PROMPTS_DIR).mkdir(parents=True, exist_ok=True)
Path(INF_GEN_OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)
Path(INF_CLASS_PROMPTS_DIR).mkdir(parents=True, exist_ok=True)
Path(INF_CLASS_OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)

### Load SSF-Corpus

In [ ]:
ssf_corpus = load_dataset(SSF_CORPUS_HF, token=True)['full']
ssf_corpus_demo_df = ssf_corpus.to_pandas().sample(frac=1, random_state=42).head(NUM_DEMO_STORIES)
reddit_corpus = Corpus(download('reddit-corpus-small'))
ssf_corpus_demo_df['text'] = ssf_corpus_demo_df['id'].apply(lambda id: reddit_corpus.get_utterance(id).text)
ssf_corpus_demo_df

### SSF-Generator

In [ ]:
taxonomy = Taxonomy(taxonomy_dir=TAXONOMY_DIR)

# prep inference generation prompts
for dim in taxonomy.get_dims():
    dim_prompts_list = []
    for i, row in ssf_corpus_demo_df.iterrows():
        prompt_text = (InferenceGenerationPromptBuilder(taxonomy=taxonomy, 
                                                        dim=dim,
                                                        text=row['text'], 
                                                        single_output=True)
                                                        .community_name(row['community'])
                                                        .community_description(row['communityDescription'])
                                                        .community_values(row['communityValues'])
                                                        .progenitor_summary(row['progenitorContext'])
                                                        .conversation_summary(row['conversationContext'])
                                                        .build())
        dim_prompts_list.append({"id": row['id'], "prompt": prompt_text})
    prompts_path = f"{INF_GEN_PROMPTS_DIR}/{dim}.jsonl"
    InferenceUtils.save_prompts_as_jsonl(dim_prompts_list, prompts_path)

# inference generation
model_config = ModelConfig(model_name=BASE_MODEL_HF)
generation_config = GenerationConfig(temperature=0.0,
                                     top_p=1.0,
                                     max_new_tokens=512,
                                     batch_size=10,
                                     max_model_len=2048)
generator_strategy = VllmGenerationStrategy(model_config=model_config,
                                            generation_config=generation_config,
                                            base_model_name=BASE_MODEL_HF,
                                            lora_adapter_path=SSF_GENERATOR_HF)
for dim in taxonomy.get_dims():
  prompts_path = f"{INF_GEN_PROMPTS_DIR}/{dim}.jsonl"
  output_path = f"{INF_GEN_OUTPUTS_DIR}/{dim}.jsonl"
  generator_strategy.generate(prompts_path, output_path)

clear_torch_memory(generator_strategy)

### SSF-Classifier

In [ ]:
# prep inference classification prompts
for dim in taxonomy.get_dims():
    inference_path = f"{INF_GEN_OUTPUTS_DIR}/{dim}.jsonl"
    inferences = InferenceUtils.read_jsonl(inference_path)
    classification_prompts = []
    for item in inferences:
        inference_text = item['output']
        prompt_builder = InferenceClassificationPromptBuilder(taxonomy=taxonomy,
                                                                             text=inference_text,
                                                                             dim=dim,
                                                                             k=0)
        classification_prompts.append({"id": item['id'], "prompt": prompt_builder.build()})
    output_path = f"{INF_CLASS_PROMPTS_DIR}/{dim}.jsonl"
    InferenceUtils.save_prompts_as_jsonl(classification_prompts, output_path)

# inference classification
classifier_strategy = VllmGenerationStrategy(
    model_config=model_config,
    generation_config=generation_config,
    base_model_name=BASE_MODEL_HF,
    lora_adapter_path=SSF_CLASSIFIER_HF
)
for dim in taxonomy.get_dims():
    input_path = f"{INF_CLASS_PROMPTS_DIR}/{dim}.jsonl"
    output_path = f"{INF_CLASS_OUTPUTS_DIR}/{dim}.jsonl"
    classifier_strategy.generate(input_path, output_path)

### Postprocess Results

In [ ]:
def parse_labels(output_text):
    parsed = InferenceUtils.parse_json(output_text.lstrip("```json").rstrip("```").strip())
    categories = parsed['response']
    return categories if isinstance(categories, list) else ([categories] if categories else [])

dims = taxonomy.get_dims()
inferences = {dim: {item['id']: item['output'] for item in InferenceUtils.read_jsonl(f"{INF_GEN_OUTPUTS_DIR}/{dim}.jsonl")}
              for dim in dims}
classifications = {dim: {item['id']: item['output'] for item in InferenceUtils.read_jsonl(f"{INF_CLASS_OUTPUTS_DIR}/{dim}.jsonl")}
                  for dim in dims}

story_ids = list(inferences[dims[0]].keys())
results_data = []
for story_id in story_ids:
    story_text = ssf_corpus_demo_df[ssf_corpus_demo_df['id'] == story_id].iloc[0]['text']
    row = {'story_id': story_id, 'text': story_text}
    for dim in dims:
        row[f'{dim}_inference'] = InferenceUtils.parse_json(inferences[dim][story_id])
        row[f'{dim}_labels'] = parse_labels(classifications[dim][story_id]) if story_id in classifications[dim] else None
    results_data.append(row)
results_df = pd.DataFrame(results_data)
output_path = f"{OUTPUTS_DIR}/ssf_demo_results.csv"
results_df.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")

In [ ]:
display(results_df)

first_result = results_df.iloc[0]
print("DETAILED RESULTS FOR FIRST STORY")
for dim in taxonomy.get_dims():
    print(f"\n[{dim.upper()}]")
    inference = first_result.get(f'{dim}_inference', 'N/A')
    labels = first_result.get(f'{dim}_labels', 'N/A')
    print(f"  Inference: {str(inference)}...")
    print(f"  Labels: {labels}")